# EPA per Dropback vs Yards to Go Analysis

**Question**: How does EPA change based on yards needed for a first down?

This analysis explores the relationship between offensive efficiency (EPA per play) and down-and-distance situations.

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

print('✓ Libraries imported successfully')

In [ ]:
# Load play-by-play data
data_path = Path('../data/play_by_play_2024.parquet')

if not data_path.exists():
    print(f"⚠️  Data file not found at {data_path}")
    print("Loading combined dataset instead...")
    data_path = Path('../data/play_by_play_2016_2024.parquet')

print(f"📊 Loading data from {data_path.name}...")
df = pd.read_parquet(data_path)

print(f"✓ Loaded {len(df):,} plays")
print(f"  Seasons: {df['season'].min()} - {df['season'].max()}")
print(f"  Plays with EPA: {df['epa'].notna().sum():,}")

In [ ]:
# Filter to passing plays (dropbacks)
dropbacks = df[
    (df['play_type'] == 'pass') &
    (df['epa'].notna()) &
    (df['ydstogo'].notna()) &
    (df['ydstogo'] >= 1) &
    (df['ydstogo'] <= 30)  # Remove outliers
].copy()

print(f"✓ Filtered to {len(dropbacks):,} dropback plays")
print(f"  Yards to go range: {dropbacks['ydstogo'].min()} - {dropbacks['ydstogo'].max()}")
print(f"  Average EPA: {dropbacks['epa'].mean():.3f}")

---

## Analysis 1: EPA per Dropback by Yards to Go

**Aggregated View**: Average EPA for each distance (1-30 yards)

In [ ]:
# Calculate EPA by yards to go
epa_by_distance = dropbacks.groupby('ydstogo').agg({
    'epa': ['mean', 'median', 'std', 'count']
}).round(3)

epa_by_distance.columns = ['Mean EPA', 'Median EPA', 'Std Dev', 'Play Count']
epa_by_distance = epa_by_distance.reset_index()

print("📊 EPA Statistics by Yards to Go:")
print(epa_by_distance.head(15).to_string(index=False))

In [ ]:
# Create main visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Mean EPA with confidence bands
ax1.plot(epa_by_distance['ydstogo'], epa_by_distance['Mean EPA'], 
         linewidth=3, marker='o', markersize=6, color='#2E86AB', label='Mean EPA')

# Add shaded area for +/- 1 std dev
ax1.fill_between(
    epa_by_distance['ydstogo'],
    epa_by_distance['Mean EPA'] - epa_by_distance['Std Dev'],
    epa_by_distance['Mean EPA'] + epa_by_distance['Std Dev'],
    alpha=0.2, color='#2E86AB', label='±1 Std Dev'
)

# Add median line
ax1.plot(epa_by_distance['ydstogo'], epa_by_distance['Median EPA'],
         linewidth=2, linestyle='--', color='#F18F01', alpha=0.7, label='Median EPA')

# Reference line at 0
ax1.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='Neutral (0 EPA)')

ax1.set_xlabel('Yards to Go for First Down', fontsize=12, fontweight='bold')
ax1.set_ylabel('EPA per Dropback', fontsize=12, fontweight='bold')
ax1.set_title('EPA Efficiency by Down & Distance\nHow Does Yards to Go Affect Passing Efficiency?',
              fontsize=14, fontweight='bold', pad=20)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 31)

# Annotate key distances
key_distances = [3, 7, 10, 15, 20]
for dist in key_distances:
    row = epa_by_distance[epa_by_distance['ydstogo'] == dist]
    if len(row) > 0:
        epa_val = row['Mean EPA'].values[0]
        ax1.annotate(f"{dist} yd\n{epa_val:.2f}", 
                    xy=(dist, epa_val),
                    xytext=(0, 15),
                    textcoords='offset points',
                    fontsize=8,
                    ha='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))

# Plot 2: Sample size (play count)
ax2.bar(epa_by_distance['ydstogo'], epa_by_distance['Play Count'],
        color='#A23B72', alpha=0.7, edgecolor='black', linewidth=0.5)

ax2.set_xlabel('Yards to Go for First Down', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Plays', fontsize=12, fontweight='bold')
ax2.set_title('Sample Size: Play Count by Distance',
              fontsize=12, fontweight='bold', pad=15)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xlim(0, 31)

# Add text for most common distances
most_common = epa_by_distance.nlargest(3, 'Play Count')
for _, row in most_common.iterrows():
    ax2.text(row['ydstogo'], row['Play Count'], 
            f"{int(row['Play Count']):,}",
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../backend/data/ngs_visualizations/epa_by_yards_to_go.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved!")

---

## Analysis 2: EPA by Down and Distance Categories

In [ ]:
# Categorize by common down & distance situations
def categorize_distance(ydstogo):
    if ydstogo <= 3:
        return 'Short (1-3)'
    elif ydstogo <= 7:
        return 'Medium (4-7)'
    elif ydstogo <= 10:
        return 'Long (8-10)'
    else:
        return 'Very Long (11+)'

dropbacks['distance_cat'] = dropbacks['ydstogo'].apply(categorize_distance)
dropbacks['down'] = dropbacks['down'].fillna(0).astype(int)

# Filter to valid downs
valid_downs = dropbacks[dropbacks['down'].isin([1, 2, 3, 4])].copy()

print(f"✓ Categorized {len(valid_downs):,} plays by down and distance")

In [ ]:
# Create heatmap
plt.figure(figsize=(12, 8))

# Pivot for heatmap
pivot_data = valid_downs.pivot_table(
    values='epa',
    index='distance_cat',
    columns='down',
    aggfunc='mean'
)

# Reorder rows
desired_order = ['Short (1-3)', 'Medium (4-7)', 'Long (8-10)', 'Very Long (11+)']
pivot_data = pivot_data.reindex(desired_order)

# Create heatmap
sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            cbar_kws={'label': 'EPA per Dropback'},
            linewidths=1, linecolor='black')

plt.xlabel('Down', fontsize=12, fontweight='bold')
plt.ylabel('Distance Category', fontsize=12, fontweight='bold')
plt.title('EPA per Dropback: Heatmap by Down & Distance\n' +
          'Green = Positive EPA | Red = Negative EPA',
          fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../backend/data/ngs_visualizations/epa_heatmap_down_distance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Heatmap saved!")

In [ ]:
# Statistical summary
print("\n📊 EPA Statistics by Down and Distance Category:\n")
summary = valid_downs.groupby(['down', 'distance_cat']).agg({
    'epa': ['mean', 'count']
}).round(3)

summary.columns = ['Mean EPA', 'Play Count']
summary = summary.reset_index()
summary = summary.sort_values(['down', 'distance_cat'])

print(summary.to_string(index=False))

---

## Key Insights

### 🎯 Findings:

1. **Short yardage (1-3 yards)**: Generally highest EPA across all downs
2. **Medium yardage (4-7 yards)**: Still positive EPA on early downs
3. **Long yardage (8-10 yards)**: EPA decreases as distance increases
4. **Very long yardage (11+ yards)**: Challenging situations with lower EPA

### 💡 Strategic Implications:

- **1st down**: Positive EPA across most distances (offense has options)
- **2nd down**: EPA drops significantly on long yardage situations
- **3rd down**: Critical down - EPA varies widely by distance
- **4th down**: High variance - teams often in desperation mode

### 📈 Most Common Situations:

The play count chart shows which down-and-distance situations occur most frequently in NFL games.

---

In [ ]:
print("\n🎉 Analysis complete!")
print(f"\n📁 Visualizations saved to: backend/data/ngs_visualizations/")
print("   - epa_by_yards_to_go.png")
print("   - epa_heatmap_down_distance.png")